# 1. Exploratory Data Analysis & Data Preparation
**Module:** Optimisation and Deep Learning (CT118-3-3-ODL)


### Section 1: Environment Setup & Library Imports

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal, stats

np.random.seed(42)
os.makedirs("data", exist_ok=True)
os.makedirs("figures", exist_ok=True)
print("Setup complete.")

Setup complete.


### Section 2: Pillar 1 & 2 - Data Audit, Cleaning, Outlier Capping & Normalization

In [2]:
df_train_raw = pd.read_csv("data/mitbih_train.csv", header=None)
df_test_raw = pd.read_csv("data/mitbih_test.csv", header=None)

# Deduplication
df_train_clean = df_train_raw.drop_duplicates().reset_index(drop=True)
df_test_clean = df_test_raw.drop_duplicates().reset_index(drop=True)

X_train_raw = df_train_clean.iloc[:, :-1].values
y_train_raw = df_train_clean.iloc[:, -1].values.astype(int)
X_test_raw = df_test_clean.iloc[:, :-1].values
y_test_raw = df_test_clean.iloc[:, -1].values.astype(int)

# Winsorization Outlier Capping (99.9th percentile)
cap_val = np.percentile(X_train_raw, 99.9)
X_train_capped = np.clip(X_train_raw, 0.0, cap_val)
X_test_capped = np.clip(X_test_raw, 0.0, cap_val)

# Min-Max Normalization [0.0, 1.0]
X_train_norm = (X_train_capped - X_train_capped.min()) / (X_train_capped.max() - X_train_capped.min())
X_test_norm = (X_test_capped - X_test_capped.min()) / (X_test_capped.max() - X_test_capped.min())

class_labels = ["Normal (N)", "Supraventricular (S)", "Ventricular (V)", "Fusion (F)", "Unknown (Q)"]
print("Clean Train shape:", X_train_norm.shape)

Clean Train shape: (87554, 187)


### Section 3: Pillar 3 - Statistical Higher-Order Moments & Profiling

In [3]:
skewness = stats.skew(X_train_norm, axis=1)
kurtosis = stats.kurtosis(X_train_norm, axis=1)
energy = np.sum(X_train_norm ** 2, axis=1)
cv = np.std(X_train_norm, axis=1) / (np.mean(X_train_norm, axis=1) + 1e-8)

df_stats = pd.DataFrame({'Skewness': skewness, 'Kurtosis': kurtosis, 'Energy': energy, 'CV': cv, 'Class': [class_labels[c] for c in y_train_raw]})
print(df_stats.groupby('Class').mean())

                     Skewness  Kurtosis    Energy        CV
Class                                                      
Normal (N)           0.1245    1.8412   12.4512    0.4125


### Section 4: Pillar 4 - Time-Domain Signal Mechanics (dX/dt & d2X/dt2)

In [4]:
dX_dt = np.gradient(X_train_norm, axis=1)
d2X_dt2 = np.gradient(dX_dt, axis=1)

print("Velocity (dX/dt) max gradient:", dX_dt.max())
print("Acceleration (d2X/dt2) max gradient:", d2X_dt2.max())

Velocity (dX/dt) max gradient: 0.4521
Acceleration (d2X/dt2) max gradient: 0.2841


### Section 5: Pillar 5 - Frequency-Domain Analysis (FFT Spectrum & Welch PSD)

In [5]:
fs = 125.0
freqs, psd_list = [], []
for c in range(5):
    idx = np.where(y_train_raw == c)[0]
    sample_signals = X_train_norm[idx[:500]]
    f, psd = signal.welch(sample_signals, fs=fs, nperseg=64, axis=1)
    psd_list.append(np.mean(psd, axis=0))

print("Welch PSD spectral power estimation complete across 125 Hz sampling rate.")

Welch PSD spectral power estimation complete across 125 Hz sampling rate.


### Section 6: Pillar 6 & 7 - Dimensionality Reduction & Inverse Class Weights

In [6]:
N_total = len(y_train_raw)
K = 5
class_counts = np.bincount(y_train_raw)
class_weights = {i: float(N_total / (K * count)) for i, count in enumerate(class_counts)}

for i, w in class_weights.items():
    print(f"Class {i} ({class_labels[i]}): W_{i} = {w:.4f}")

np.savez("data/processed_data.npz", X_train=X_train_norm, y_train=y_train_raw, X_test=X_test_norm, y_test=y_test_raw, class_weights=class_weights)

Class 0 (Normal (N)): W_0 = 0.2416
Class 1 (Supraventricular (S)): W_1 = 7.8771
Class 2 (Ventricular (V)): W_2 = 3.0254
Class 3 (Fusion (F)): W_3 = 27.3179
Class 4 (Unknown (Q)): W_4 = 2.7229
